# BBO Capstone Project — Bayesian Optimization Challenge

**Imperial College London — Professional Certificate in Machine Learning and AI**

This notebook supports Stage 2 of the capstone project: an iterative Black-Box
Optimization (BBO) challenge across eight unknown functions of increasing
dimensionality (2D to 8D). Each week you submit one query point per function
via the capstone project portal, and receive back one new output value per
function to grow your dataset.

**What this notebook contains:**
1. Setup (installing dependencies)
2. Exploratory Data Analysis (EDA) of the initial datasets provided for each function
3. A Bayesian Optimization helper (ask–tell interface) to decide the next query
   point for each function every week
4. A weekly workflow section to run each time you have a new round of results

**Folder structure expected next to this notebook:**
```
your_folder/
├── BBO_Capstone_Notebook.ipynb   (this file)
├── analyze_initial_data.py
├── bbo_optimizer.py
├── initial_data/
│   ├── function_1/
│   │   ├── initial_inputs.npy
│   │   └── initial_outputs.npy
│   ├── function_2/
│   ...
└── bbo_state/                     (created automatically to store your weekly results)
```


## How to read this notebook

**If you are new to this project, read this page first. It explains the whole thing in
plain language before any code appears.**

### The problem in one paragraph

I am given eight hidden mathematical functions. I cannot see inside any of them. I can
only send in a set of numbers and get back a single score, and my goal is to find the
input that produces the highest score for each function. The catch is the budget: I am
allowed one attempt per function per week, for thirteen weeks. Each function also came
with a small set of starting examples. So every single query has to count.

### The method in one paragraph

I use a technique called Bayesian Optimisation. It works in two parts. A model called a
Gaussian Process looks at everything observed so far and builds a best guess of what the
hidden function looks like, together with an honest statement of how confident it is in
each region. A second piece, called an acquisition function, reads that guess and picks
where to look next, balancing two instincts: go back to where results were already good,
or go somewhere unknown where something better might be hiding.

### How the notebook is organised

| Section | What it is for | Do you need to read the code? |
|---|---|---|
| 1. Setup | Installs the libraries | No |
| 2. Exploratory data analysis | Looks at the starting data for each function | No, just run it |
| 3. Optimisation engine | The core method that chooses each week's query | No, but it is documented |
| 4. Weekly workflow | **The part I actually use every week** | Yes, this is the short bit |
| 6. Comparing results | Checks whether a query was good or bad | No, just run it |
| 8 to 10. Extra analysis | Secondary checks used to understand the functions | No, just run it |

### Important note about the long code cells

Four cells in this notebook are long: sections 2, 3, 8 and 9. **They are reference
implementations, not cells you need to read to follow the project.** Each one defines a
set of named functions, and then a short cell immediately afterwards calls them. The
markdown box above each long cell explains in plain words what it does and which
commands it gives you.

If you only want to understand what I do each week, skip to **section 4**. That part is
six lines of code.

### The three commands that matter

Everything in the weekly routine comes down to three function calls:

- `suggest_all()` asks the model for the next point to try, for all eight functions
- `record(n, result)` tells the model what score came back for function `n`
- `summary()` shows where each function stands and how much budget is left


## 1. Setup

Run this once. If the packages are already installed, it will simply confirm that.

In [ ]:
import sys
!{sys.executable} -m pip install scikit-optimize numpy matplotlib

**Important:** after running the install cell above for the first time, restart the kernel
(Kernel → Restart Kernel) before running the cells below, so Python picks up the newly
installed packages.

## 2. Exploratory data analysis of the starting data

### What this section does, in plain words

Before optimising anything, it is worth looking at what I was given. The course provided
a set of starting examples for each of the eight functions: pairs of "here is an input,
here is the score it produced". The amount varies by how complicated the function is,
from 10 examples for the simplest up to 40 for the hardest.

This section loads those examples and answers three basic questions for each function:

1. **How much data do I have, and over what range?** A function with 40 examples spread
   across eight inputs is far more thinly covered than one with 10 examples across two.
2. **What do the scores look like?** The highest, lowest and average score tells me what
   counts as a good result before I have tried anything myself.
3. **Does any single input obviously drive the score?** The plots show each input against
   the score, so an obvious trend would be visible straight away.

### What the long cell below contains

It defines four named commands. You do not need to read the code to use them:

- `summary_all()` prints the statistics described above for all eight functions
- `plot_all()` draws the charts for all eight functions and saves them as image files
- `print_summary(n)` and `plot_function(n)` do the same for one function at a time

The two short cells after it are what actually produce the output.


In [ ]:
"""
Exploratory Data Analysis (EDA) of the BBO capstone initial datasets
========================================================================
Loads the .npy files provided by the course (initial_data/function_N/) for the
eight black-box functions and produces:
- Summary statistics (shape, X range, y range/mean/std)
- Scatter plots to visualise where the best points are so far
- An input-vs-output correlation view for functions with more than 2 dimensions

Requirements: numpy, matplotlib
Usage: place this file in the same folder as initial_data/ and run
       python analyze_initial_data.py
or import the functions in a Jupyter notebook.
"""

import os
import numpy as np
import matplotlib.pyplot as plt

INITIAL_DATA_DIR = "initial_data"
FUNCTION_DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}
OUTPUT_DIR = "eda_plots"


def load_function_data(func_id):
    """Load X (input) and y (output) for a given function."""
    folder = os.path.join(INITIAL_DATA_DIR, f"function_{func_id}")
    X = np.load(os.path.join(folder, "initial_inputs.npy"))
    y = np.load(os.path.join(folder, "initial_outputs.npy"))
    return X, y


def print_summary(func_id):
    """Print summary statistics for a given function."""
    X, y = load_function_data(func_id)
    dim = FUNCTION_DIMS[func_id]
    best_idx = int(np.argmax(y))

    print(f"\n{'='*60}")
    print(f"Function {func_id}  ({dim}-D)")
    print(f"{'='*60}")
    print(f"Number of initial points : {len(y)}")
    print(f"X range per dimension:")
    for d in range(dim):
        print(f"    x{d + 1}: [{X[:, d].min():.4f}, {X[:, d].max():.4f}]")
    print(f"Output y  -> min={y.min():.4f}  max={y.max():.4f}  "
          f"mean={y.mean():.4f}  std={y.std():.4f}")
    print(f"Best point so far        : x={np.round(X[best_idx], 4).tolist()}  "
          f"-> y={y[best_idx]:.4f}")


def summary_all():
    """Print statistics for all 8 functions."""
    for func_id in FUNCTION_DIMS:
        print_summary(func_id)


def plot_function(func_id, save=True):
    """
    Generate a plot for a given function:
    - If 2D: scatter plot x1 vs x2, coloured by y (shows where the high-output
      regions are)
    - If >2D: a grid of scatter plots, one per dimension vs y, to see which
      variables seem to influence the output the most
    """
    X, y = load_function_data(func_id)
    dim = FUNCTION_DIMS[func_id]
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if dim == 2:
        fig, ax = plt.subplots(figsize=(6, 5))
        sc = ax.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", s=80, edgecolor="k")
        best_idx = int(np.argmax(y))
        ax.scatter(X[best_idx, 0], X[best_idx, 1], c="red", s=200,
                   marker="*", edgecolor="k", label="Best point so far")
        ax.set_xlabel("x1")
        ax.set_ylabel("x2")
        ax.set_title(f"Function {func_id} ({dim}-D) — initial points")
        ax.legend()
        plt.colorbar(sc, label="y (output)")
        plt.tight_layout()
    else:
        cols = min(dim, 4)
        rows = int(np.ceil(dim / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.5 * rows))
        axes = np.array(axes).reshape(-1)
        for d in range(dim):
            ax = axes[d]
            ax.scatter(X[:, d], y, alpha=0.7, edgecolor="k")
            ax.set_xlabel(f"x{d + 1}")
            ax.set_ylabel("y")
            ax.set_title(f"x{d + 1} vs y")
        for d in range(dim, len(axes)):
            axes[d].axis("off")
        fig.suptitle(f"Function {func_id} ({dim}-D) — each variable vs output")
        plt.tight_layout()

    if save:
        path = os.path.join(OUTPUT_DIR, f"function_{func_id}.png")
        plt.savefig(path, dpi=120)
        print(f"Saved: {path}")
    plt.show()


def plot_all():
    """Generate plots for all 8 functions."""
    for func_id in FUNCTION_DIMS:
        plot_function(func_id)


if __name__ == "__main__":
    summary_all()
    plot_all()


### Running it

`summary_all()` prints a block of statistics per function. The line worth looking at is
"Best point so far", which is the highest score in the starting data. That is the number
each week's query has to beat to count as progress.


In [ ]:
summary_all()

In [ ]:
plot_all()

## 3. The optimisation engine

### What this section does, in plain words

This is the part that decides where to look next. It follows a pattern called **ask and
tell**, which works exactly as it sounds:

- **Ask**: the model proposes a point to try. I copy it into the course portal.
- **Tell**: about a week later the portal emails me the score. I feed it back in.

The model then rebuilds itself from the full history and is ready for the next ask. Over
thirteen rounds it gradually learns the shape of each hidden function.

### The two pieces of the method

**The Gaussian Process** is the model. Given every point observed so far, it produces a
best guess of the score at any other point in the space, plus a measure of how uncertain
that guess is. Near points I have already tested it is confident. Far away it is not,
and it says so.

**Expected Improvement** is the rule that turns that guess into a decision. For every
candidate point it asks: how much better than my current best could this realistically
be? A point the model is confident about but which looks average scores poorly. A point
the model knows nothing about scores well, because it might surprise. This is how the
balance between playing safe and taking a risk happens automatically, without me tuning
a dial.

### Two practical details worth knowing

**The score is flipped internally.** The library this is built on is designed to find
the lowest value, while all eight of these functions reward the highest. So the code
negates every score before handing it to the library, and negates it back when reporting
results. Everything you see in the output is the real, unflipped score.

**Progress is saved to disk.** Every result is written to a small file in a folder called
`bbo_state`, one file per function. This means I can close the notebook, come back a week
later, and the model rebuilds itself from the full history. Nothing depends on anything
staying open in memory, which matters when the project runs over three months.

### What the long cell below gives you

Again, these are named commands rather than code you need to read:

| Command | What it does |
|---|---|
| `suggest_all()` | Proposes the next point for all eight functions |
| `suggest(n)` | Proposes the next point for function `n` only |
| `record(n, score, x=[...])` | Saves the score that came back for function `n` |
| `summary()` | Shows the state of all eight functions and the budget left |
| `best(n)` | Shows the best result found so far for function `n` |
| `undo_last_all()` | Removes the last saved result, to correct a mistake |
| `final_summary()` | One closing table for the whole project (section 11) |


In [ ]:
"""
BBO Capstone Optimizer
=======================
Ask-tell Bayesian Optimization helper for the Imperial ML Capstone BBO challenge.

Weekly workflow, for each of the 8 functions:
1. Call `suggest(func_id)` to get the next point to try.
2. Copy the printed string into the "Function N" field on the capstone project
   portal and submit.
3. Once you get the output value back from the platform, call
   `record(func_id, y_value)` to update the model.
4. Repeat. The more queries you make, the better the model learns the shape
   of the function.

State (input/output history) is saved to disk in the bbo_state/ folder, so you
can close and reopen Python without losing progress.

Requirements: pip install scikit-optimize numpy
"""

import json
import os
import numpy as np
from skopt import Optimizer
from skopt.space import Real

STATE_DIR = "bbo_state"
os.makedirs(STATE_DIR, exist_ok=True)

# Folder with the real initial data provided by the course
# (initial_data/function_N/initial_inputs.npy, initial_outputs.npy).
# Extract the "Initial_data_points_starter.zip" here so the structure is:
# initial_data/function_1/, function_2/, etc.
INITIAL_DATA_DIR = "initial_data"

# Dimensionality of each of the 8 functions, as shown on the platform
FUNCTION_DIMS = {
    1: 2,
    2: 2,
    3: 3,
    4: 4,
    5: 4,
    6: 5,
    7: 6,
    8: 8,
}

# Assumption: domain [0, 1] for every variable, based on the initial data
# provided and the platform's input placeholder ("0.000000-0.000000").
# If you discover the real domain differs, change it here.
BOUNDS = (0.0, 1.0)

# The portal has 13 total submission weeks (Module 12 -> Module 24), one
# submission per week covering all 8 functions together.
# So the budget is 13 total queries PER function, not shared across all of them.
TOTAL_WEEKS = 13

_optimizers = {}
_pending = {}  # func_id -> last suggested point, waiting for a result


def _state_path(func_id):
    return os.path.join(STATE_DIR, f"function_{func_id}.json")


def _load_initial_data(func_id):
    """Load the initial (x, y) points provided by the course for a function, if present."""
    folder = os.path.join(INITIAL_DATA_DIR, f"function_{func_id}")
    x_path = os.path.join(folder, "initial_inputs.npy")
    y_path = os.path.join(folder, "initial_outputs.npy")
    if os.path.exists(x_path) and os.path.exists(y_path):
        X = np.load(x_path).tolist()
        y = np.load(y_path).tolist()
        return X, y
    return [], []


def _load_history(func_id):
    """
    Load the history for a function: if no saved state exists yet, initialise
    it with the real initial data provided by the course (initial_data/).
    Subsequent weekly queries are then accumulated on top of this data.
    """
    path = _state_path(func_id)
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        return data.get("X", []), data.get("y", [])

    # First time: start from the real initial data, if available
    X, y = _load_initial_data(func_id)
    if X:
        _save_history(func_id, X, y)
        print(f"Function {func_id}: loaded {len(y)} initial points from {INITIAL_DATA_DIR}/function_{func_id}/")
    return X, y


def _save_history(func_id, X, y):
    with open(_state_path(func_id), "w") as f:
        json.dump({"X": X, "y": y}, f, indent=2)


def _get_optimizer(func_id):
    if func_id not in _optimizers:
        dim = FUNCTION_DIMS[func_id]
        space = [Real(*BOUNDS) for _ in range(dim)]
        X, y = _load_history(func_id)
        # We already have real initial data (from the course or previous
        # queries), so n_initial_points=0: the optimizer uses the GP model
        # right away instead of proposing more random/Sobol points.
        n_initial = 0 if X else 4
        opt = Optimizer(
            dimensions=space,
            base_estimator="GP",
            acq_func="EI",  # Expected Improvement
            acq_optimizer="auto",
            n_initial_points=n_initial,
            initial_point_generator="sobol",
            random_state=42 + func_id,  # different seed per function, otherwise Sobol sequences would coincide across functions
        )
        if X:
            # skopt minimises internally -> we negate y because we want to MAXIMISE
            opt.tell(X, [-v for v in y])
        _optimizers[func_id] = opt
    return _optimizers[func_id]


def suggest(func_id):
    """Return the next point to try for a given function, formatted for the platform."""
    opt = _get_optimizer(func_id)
    x = opt.ask()
    _pending[func_id] = x
    formatted = "-".join(f"{v:.6f}" for v in x)
    print(f"Function {func_id} ({FUNCTION_DIMS[func_id]}-D) -> submit this value:")
    print(formatted)
    return formatted


def record(func_id, y_value, x=None):
    """
    Record the result returned by the platform for the last suggested point
    (or an explicit x, if you're re-entering a past result)."""
    opt = _get_optimizer(func_id)
    x = x if x is not None else _pending.get(func_id)
    if x is None:
        raise ValueError(
            f"No pending suggestion for function {func_id}. "
            f"Call suggest({func_id}) first, or pass x explicitly."
        )
    opt.tell(x, -y_value)  # negated because skopt minimises, we want to maximise

    X, y = _load_history(func_id)
    X.append(list(x))
    y.append(y_value)
    _save_history(func_id, X, y)
    _pending.pop(func_id, None)

    best_idx = int(np.argmax(y))
    print(f"Recorded. Best result so far for function {func_id}: "
          f"y={y[best_idx]:.6f} at x={X[best_idx]}")


def best(func_id):
    """Show the best result found so far for a function."""
    X, y = _load_history(func_id)
    if not y:
        print(f"No data yet for function {func_id}.")
        return None
    idx = int(np.argmax(y))
    print(f"Function {func_id}: best y={y[idx]:.6f} at x={X[idx]}")
    return X[idx], y[idx]


def undo_last(func_id):
    """
    Remove the most recently recorded point for a function -- useful if you
    accidentally recorded a duplicate or wrong result. Refuses to remove one of
    the original initial data points (only removes weekly-recorded entries).
    """
    X, y = _load_history(func_id)
    initial_X, _ = _load_initial_data(func_id)
    n_initial = len(initial_X)
    if len(y) <= n_initial:
        print(f"Function {func_id}: nothing to undo (only initial data present).")
        return
    removed_x, removed_y = X.pop(), y.pop()
    _save_history(func_id, X, y)
    _optimizers.pop(func_id, None)  # force the optimizer to rebuild from the corrected history
    print(f"Function {func_id}: removed last entry x={removed_x} -> y={removed_y}")


def undo_last_all():
    """Remove the most recently recorded point for all 8 functions."""
    for func_id in FUNCTION_DIMS:
        undo_last(func_id)


def summary():
    """Print a summary of all 8 functions: initial points, weekly queries used/remaining, best result."""
    for func_id in FUNCTION_DIMS:
        X, y = _load_history(func_id)
        initial_X, _ = _load_initial_data(func_id)
        n_initial = len(initial_X)
        n_weekly = len(y) - n_initial
        remaining = TOTAL_WEEKS - n_weekly
        if not y:
            print(f"Function {func_id}: no data yet")
        else:
            idx = int(np.argmax(y))
            print(f"Function {func_id}: {n_initial} initial points + {n_weekly}/{TOTAL_WEEKS} weekly queries "
                  f"({remaining} remaining), best y={y[idx]:.6f}")


def suggest_all():
    """Generate the suggested point for all 8 functions at once, since the
    weekly submission requires all functions together."""
    results = {}
    for func_id in FUNCTION_DIMS:
        results[func_id] = suggest(func_id)
    return results


def compare_to_initial(func_id):
    """
    Compare each weekly query's output against the initial dataset's distribution.
    Prints, for every weekly point, its percentile rank relative to the initial y
    values -- e.g. "better than 80% of the initial points" -- so you can judge how
    good a guess was without knowing the true global maximum.
    """
    initial_X, initial_y = _load_initial_data(func_id)
    X, y = _load_history(func_id)
    n_initial = len(initial_y)
    weekly_X, weekly_y = X[n_initial:], y[n_initial:]

    if not weekly_y:
        print(f"Function {func_id}: no weekly queries recorded yet.")
        return

    initial_y_arr = np.array(initial_y)
    print(f"\nFunction {func_id} -- initial data: "
          f"min={initial_y_arr.min():.4f}, max={initial_y_arr.max():.4f}, "
          f"mean={initial_y_arr.mean():.4f}")
    for week_num, (x_val, y_val) in enumerate(zip(weekly_X, weekly_y), start=1):
        percentile = 100 * np.mean(initial_y_arr <= y_val)
        verdict = (
            "better than the best initial point" if y_val > initial_y_arr.max() else
            "worse than every initial point" if y_val < initial_y_arr.min() else
            f"better than {percentile:.0f}% of initial points"
        )
        print(f"  Week {week_num}: x={np.round(x_val, 4).tolist()} -> y={y_val:.6f}  ({verdict})")


def compare_all():
    """Run compare_to_initial for all 8 functions."""
    for func_id in FUNCTION_DIMS:
        compare_to_initial(func_id)


def plot_progress(func_id, save=True):
    """
    Plot the initial data together with the weekly queries made so far, so you can
    see visually whether your weekly points are landing in strong or weak regions.
    - If 2D: scatter plot x1 vs x2, coloured by y. Initial points are circles,
      weekly points are stars (numbered by week).
    - If >2D: one scatter per input dimension vs y, with weekly points highlighted.
    """
    import matplotlib.pyplot as plt
    import os as _os

    initial_X, initial_y = _load_initial_data(func_id)
    X, y = _load_history(func_id)
    dim = FUNCTION_DIMS[func_id]
    n_initial = len(initial_y)
    weekly_X, weekly_y = np.array(X[n_initial:]), y[n_initial:]
    initial_X, initial_y = np.array(initial_X), np.array(initial_y)

    _os.makedirs("eda_plots", exist_ok=True)

    if dim == 2:
        fig, ax = plt.subplots(figsize=(6, 5))
        sc = ax.scatter(initial_X[:, 0], initial_X[:, 1], c=initial_y, cmap="viridis",
                        s=80, edgecolor="k", label="Initial data")
        if len(weekly_y):
            ax.scatter(weekly_X[:, 0], weekly_X[:, 1], c=weekly_y, cmap="viridis",
                      s=250, marker="*", edgecolor="red", linewidth=1.5, label="Weekly queries")
            for i, (xr, yr) in enumerate(zip(weekly_X, weekly_y), start=1):
                ax.annotate(f"W{i}", (xr[0], xr[1]), textcoords="offset points",
                           xytext=(6, 6), fontsize=9, color="red")
        ax.set_xlabel("x1")
        ax.set_ylabel("x2")
        ax.set_title(f"Function {func_id} ({dim}-D) -- initial data vs weekly queries")
        ax.legend()
        plt.colorbar(sc, label="y (output)")
        plt.tight_layout()
    else:
        cols = min(dim, 4)
        rows = int(np.ceil(dim / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.5 * rows))
        axes = np.array(axes).reshape(-1)
        for d in range(dim):
            ax = axes[d]
            ax.scatter(initial_X[:, d], initial_y, alpha=0.6, edgecolor="k", label="Initial")
            if len(weekly_y):
                ax.scatter(weekly_X[:, d], weekly_y, c="red", s=150, marker="*",
                          edgecolor="k", label="Weekly")
            ax.set_xlabel(f"x{d + 1}")
            ax.set_ylabel("y")
            ax.set_title(f"x{d + 1} vs y")
        axes[0].legend()
        for d in range(dim, len(axes)):
            axes[d].axis("off")
        fig.suptitle(f"Function {func_id} ({dim}-D) -- initial data vs weekly queries")
        plt.tight_layout()

    if save:
        path = f"eda_plots/function_{func_id}_progress.png"
        plt.savefig(path, dpi=120)
        print(f"Saved: {path}")
    plt.show()


def plot_progress_all():
    """Generate the progress plot for all 8 functions."""
    for func_id in FUNCTION_DIMS:
        plot_progress(func_id)


def final_summary():
    """
    Print one closing table for the whole project: for each function, the best
    score that was already present in the course-provided starting data, the
    best score reached overall, and which weekly round produced the record.

    Reading the table:
    - "Best initial" is the bar each weekly query had to beat.
    - "Best reached" is the highest score across starting data and weekly queries.
    - "Record week" is the weekly round that produced it, or "initial" if no
      weekly query ever beat the starting data.
    """
    header = (f"{'Func':>4}  {'Dim':>3}  {'Best initial':>14}  "
              f"{'Best reached':>14}  {'Record week':>12}  {'Improved':>9}")
    print(header)
    print("-" * len(header))

    improved_count = 0

    for func_id in FUNCTION_DIMS:
        X, y = _load_history(func_id)
        initial_X, initial_y = _load_initial_data(func_id)
        dim = FUNCTION_DIMS[func_id]

        if not y:
            print(f"{func_id:>4}  {dim:>3}  {'no data yet':>14}")
            continue

        n_initial = len(initial_y)
        best_initial = max(initial_y) if initial_y else float("-inf")
        best_overall = max(y)
        best_idx = int(np.argmax(y))

        # Which round produced the record? Weekly rounds start after the
        # initial points, so index n_initial is week 1.
        if best_idx < n_initial:
            record_week = "initial"
            improved = "no"
        else:
            record_week = str(best_idx - n_initial + 1)
            improved = "yes"
            improved_count += 1

        init_str = f"{best_initial:.4f}" if initial_y else "n/a"
        print(f"{func_id:>4}  {dim:>3}  {init_str:>14}  "
              f"{best_overall:>14.4f}  {record_week:>12}  {improved:>9}")

    print("-" * len(header))
    print(f"Functions where a weekly query beat the starting data: "
          f"{improved_count}/{len(FUNCTION_DIMS)}")


if __name__ == "__main__":
    print("BBO Capstone Optimizer loaded. Available functions:")
    print("  suggest(func_id)         -> get the next point to submit")
    print("  record(func_id, y_value) -> record the platform's result")
    print("  best(func_id)            -> show the best result so far")
    print("  summary()                -> overview of all functions")


## 4. The weekly workflow

**This is the only part of the notebook I touch during a normal week.** Everything above
sets the machinery up; everything below is extra analysis. The routine is six steps and
takes a few minutes.

### Step 1: check where things stand

`summary()` prints one line per function showing how many starting examples it has, how
many of my thirteen weekly queries I have used, and the best score so far. I run it
before anything else so I know what I am working with.


In [ ]:
summary()

### Step 2: get this week's suggestions

`suggest_all()` prints eight values, one per function, already formatted the way the
portal expects them.

**One thing to check before submitting.** Occasionally the model proposes a point it has
already tested, and the library quietly replaces it with a random one. If that random
point goes in, the week is largely wasted. So I compare the suggestions against recent
history before copying them across. This has caught a genuine duplicate more than once.


In [ ]:
suggest_all()

### Steps 3 and 4: submit, then record what came back

I copy each value into the matching "Function N" field in the course portal and submit.
About a week later an email arrives with the score each function produced.

Then I fill in the cell below with those scores and run it **once**. Note the `x=[...]`
part: it states explicitly which input produced that score. This matters because the
notebook may have been closed and reopened between submitting and recording, in which
case the model no longer remembers what it proposed. Stating the input removes any doubt.

Running this cell twice by accident saves the same result twice and wastes a query from
the budget. If that happens, `undo_last_all()` undoes it.


In [ ]:
record(1, 0.0)   # replace 0.0 with the real output for Function 1
record(2, 0.0)   # replace 0.0 with the real output for Function 2
record(3, 0.0)   # replace 0.0 with the real output for Function 3
record(4, 0.0)   # replace 0.0 with the real output for Function 4
record(5, 0.0)   # replace 0.0 with the real output for Function 5
record(6, 0.0)   # replace 0.0 with the real output for Function 6
record(7, 0.0)   # replace 0.0 with the real output for Function 7
record(8, 0.0)   # replace 0.0 with the real output for Function 8

### Step 6: confirm it saved correctly

Running `summary()` again is a deliberate check rather than a formality. The count of
used queries should have gone up by exactly one for each function. If it went up by two,
the recording cell ran twice and needs undoing.


In [ ]:
summary()

## 6. Was this week's result any good?

### The difficulty

There is no answer key. I never find out the true best score for any function, so I
cannot say "this result is 80% of the way there". The only honest comparison is against
what I already knew.

### What these two commands do

`compare_all()` takes each weekly result and places it against the spread of the starting
data for that function. It reports something like "better than 80% of the starting
examples", or better still, "better than every starting example". It is a plain way of
asking whether a query landed somewhere genuinely promising.

`plot_progress_all()` shows the same thing visually: the starting examples as ordinary
dots, and my weekly queries as red stars labelled by week number. If the stars drift
towards the high scoring region over time, the method is working.

### One caution when reading the output

These comparisons are against the **starting data**, not against last week. A result can
sit in the 90th percentile of the starting data and still be worse than the previous
week's. Both comparisons matter, so I check them separately.


In [ ]:
compare_all()

In [ ]:
plot_progress_all()

## 7. Notes and assumptions

- **Input domain:** assumed to be `[0, 1]` for every variable in every function, based on
  the initial data provided and the platform's input format (six decimal places,
  starting with `0.`).
- **Objective:** all eight functions are maximisation problems (as stated in the course
  materials); the code internally negates values where needed because `scikit-optimize`
  minimises by default.
- **Budget:** 13 weekly queries per function (Module 12 to Module 24), on top of the
  initial dataset already provided (10 to 40 points depending on the function).
- **Acquisition function:** Expected Improvement (EI), a standard, well-established
  choice that balances exploitation (querying near known good points) against
  exploration (querying uncertain regions that could hide a better optimum).


## 8. A simpler model, used to understand what is happening

### Why bother with a second model

The Gaussian Process is good at choosing where to look next, but it is poor at explaining
itself. It will not tell me which of the eight inputs actually matter, and that is
something I want to know.

So I fit a much simpler model alongside it, purely as a diagnostic. It plays no part in
choosing queries. It exists to answer questions the main model cannot.

### The trick: turn the problem into a yes or no question

A Support Vector Machine is a classifier, so it needs categories rather than scores. I
create them by splitting the observations in half at the middle score: everything above
is labelled "good", everything below "poor". The model then tries to draw a boundary
separating the two groups.

### What this tells me

**Is the function simple or complicated?** I try three different kinds of boundary: a
straight one and two curved ones. If a curved boundary separates good from poor much
better than a straight one, the function genuinely curves, which tells me its inputs
interact rather than each contributing independently. For one of the eight functions the
curved version scored 0.88 against 0.71 for the straight one, which is a clear answer.

**Which inputs matter most?** The straight boundary comes with one number per input
showing how much that input shifts the decision. Ranking those numbers gives a rough
order of importance. For the hardest function, three inputs stood out and one was close
to irrelevant.

### A limit worth stating

Only two of the eight functions have few enough inputs to draw a picture of the boundary.
For the rest, the boundary exists in a space with more dimensions than can be drawn on
paper, so I have the numbers but not the picture.


In [ ]:
"""
SVM analysis of the BBO capstone functions
========================================================================
Uses a Support Vector Machine classifier as a fast, interpretable secondary
check alongside the Gaussian Process model: labels each observed point as
"high" or "low" output (above/below the median seen so far), fits an SVM to
separate the two classes, and compares kernels to get a sense of whether the
function's response surface looks linear or non-linear in the region explored
so far.

This does NOT replace the GP/Bayesian Optimization approach -- it's a
diagnostic tool to build intuition about each function's shape and about
which input dimensions seem to matter most.

Requirements: numpy, scikit-learn
Usage: place this file next to bbo_optimizer.py (it reuses its data-loading
logic) and import the functions in the notebook.
"""

import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

from bbo_optimizer import _load_history, FUNCTION_DIMS


def _prepare_classification_data(func_id):
    """Load all (initial + weekly) points for a function and label them
    high/low based on the median output observed so far."""
    X, y = _load_history(func_id)
    X = np.array(X)
    y = np.array(y)
    median = np.median(y)
    labels = (y > median).astype(int)  # 1 = high output, 0 = low output
    return X, y, labels


def svm_kernel_comparison(func_id, kernels=("linear", "rbf", "poly")):
    """
    Fit an SVM classifier (high vs low output) for a function using different
    kernels, and report cross-validated accuracy for each. Higher accuracy
    with a non-linear kernel (rbf/poly) relative to the linear kernel suggests
    the response surface has real curvature or interaction effects between
    inputs, rather than a simple additive/linear structure.
    """
    X, y, labels = _prepare_classification_data(func_id)
    n_points = len(labels)
    n_classes = len(np.unique(labels))

    print(f"\nFunction {func_id} ({FUNCTION_DIMS[func_id]}-D) -- {n_points} points, "
          f"{n_classes} classes after median split")

    if n_points < 6 or n_classes < 2:
        print("  Not enough data yet for a meaningful SVM comparison (need >= 6 points and both classes).")
        return None

    # Standardise inputs: not strictly necessary since inputs are already in
    # [0, 1], but keeps the SVM well-behaved and comparable across functions.
    X_scaled = StandardScaler().fit_transform(X)
    cv_folds = min(5, n_points // 2, np.bincount(labels).min())
    cv_folds = max(cv_folds, 2)

    results = {}
    for kernel in kernels:
        try:
            clf = SVC(kernel=kernel, C=1.0)
            scores = cross_val_score(clf, X_scaled, labels, cv=cv_folds)
            results[kernel] = scores.mean()
            print(f"  {kernel:8s} kernel -> mean CV accuracy = {scores.mean():.3f} "
                  f"(+/- {scores.std():.3f}, {cv_folds}-fold)")
        except Exception as e:
            print(f"  {kernel:8s} kernel -> could not evaluate ({e})")
    return results


def svm_kernel_comparison_all():
    """Run the kernel comparison for all 8 functions."""
    for func_id in FUNCTION_DIMS:
        svm_kernel_comparison(func_id)


def svm_feature_importance(func_id):
    """
    Fit a linear-kernel SVM and inspect the absolute coefficient magnitude
    per input dimension, as a fast proxy for "which dimensions matter most"
    -- useful for higher-dimensional functions where the GP's reasoning is
    harder to inspect directly.
    """
    X, y, labels = _prepare_classification_data(func_id)
    n_points = len(labels)
    n_classes = len(np.unique(labels))
    dim = FUNCTION_DIMS[func_id]

    if n_points < 6 or n_classes < 2:
        print(f"Function {func_id}: not enough data yet for feature importance.")
        return None

    X_scaled = StandardScaler().fit_transform(X)
    clf = SVC(kernel="linear", C=1.0)
    clf.fit(X_scaled, labels)

    importance = np.abs(clf.coef_[0])
    order = np.argsort(importance)[::-1]

    print(f"\nFunction {func_id} ({dim}-D) -- feature importance (linear SVM coefficients):")
    for rank, idx in enumerate(order, start=1):
        print(f"  #{rank}: x{idx + 1}  |coef|={importance[idx]:.3f}")
    return importance


def svm_feature_importance_all():
    """Run feature importance analysis for all 8 functions."""
    for func_id in FUNCTION_DIMS:
        svm_feature_importance(func_id)


def plot_kernel_comparison(save=True):
    """
    Grouped bar chart comparing linear / RBF / polynomial kernel accuracy
    across all 8 functions, so the pattern (which functions look linear vs
    non-linear) is visible at a glance instead of reading a table of numbers.
    """
    import matplotlib.pyplot as plt
    import os

    kernels = ("linear", "rbf", "poly")
    all_scores = {k: [] for k in kernels}
    labels_x = []

    for func_id in FUNCTION_DIMS:
        results = svm_kernel_comparison(func_id) or {}
        labels_x.append(f"F{func_id}")
        for k in kernels:
            all_scores[k].append(results.get(k, 0))

    x = np.arange(len(labels_x))
    width = 0.25

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, k in enumerate(kernels):
        ax.bar(x + (i - 1) * width, all_scores[k], width, label=k)

    ax.set_xticks(x)
    ax.set_xticklabels(labels_x)
    ax.set_ylabel("Mean CV accuracy")
    ax.set_ylim(0, 1)
    ax.set_title("SVM kernel comparison across all 8 functions")
    ax.legend(title="Kernel")
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, alpha=0.6)
    plt.tight_layout()

    if save:
        os.makedirs("eda_plots", exist_ok=True)
        path = "eda_plots/svm_kernel_comparison.png"
        plt.savefig(path, dpi=120)
        print(f"Saved: {path}")
    plt.show()


def plot_feature_importance_all(save=True):
    """
    Grid of horizontal bar charts, one per function, showing input dimensions
    ranked by absolute linear-SVM coefficient -- a quick visual of which
    dimensions matter most for each function.
    """
    import matplotlib.pyplot as plt
    import os

    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes = axes.reshape(-1)

    for i, func_id in enumerate(FUNCTION_DIMS):
        importance = svm_feature_importance(func_id)
        ax = axes[i]
        if importance is None:
            ax.axis("off")
            continue
        dim = FUNCTION_DIMS[func_id]
        order = np.argsort(importance)
        labels = [f"x{d + 1}" for d in order]
        ax.barh(labels, importance[order], color="steelblue")
        ax.set_title(f"Function {func_id} ({dim}-D)")
        ax.set_xlabel("|coefficient|")

    plt.tight_layout()
    if save:
        os.makedirs("eda_plots", exist_ok=True)
        path = "eda_plots/svm_feature_importance.png"
        plt.savefig(path, dpi=120)
        print(f"Saved: {path}")
    plt.show()


def plot_decision_boundary(func_id, kernel="rbf", save=True):
    """
    Plot the actual data points (coloured by high/low class) together with the
    SVM's decision boundary. Only meaningful for 2D functions (1 and 2) --
    a decision boundary cannot be faithfully drawn on paper for higher
    dimensions, so this raises an error if called on a function with more
    than 2 input dimensions.
    """
    import matplotlib.pyplot as plt
    import os

    dim = FUNCTION_DIMS[func_id]
    if dim != 2:
        raise ValueError(
            f"Function {func_id} has {dim} dimensions -- decision boundary "
            f"plots are only supported for 2D functions (1 and 2)."
        )

    X, y, labels = _prepare_classification_data(func_id)
    scaler = StandardScaler().fit(X)
    X_scaled = scaler.transform(X)

    clf = SVC(kernel=kernel, C=1.0)
    clf.fit(X_scaled, labels)

    # Build a grid over the actual [0, 1] x [0, 1] input domain
    grid_n = 200
    x1_grid = np.linspace(0, 1, grid_n)
    x2_grid = np.linspace(0, 1, grid_n)
    xx1, xx2 = np.meshgrid(x1_grid, x2_grid)
    grid_points = np.column_stack([xx1.ravel(), xx2.ravel()])
    grid_scaled = scaler.transform(grid_points)
    preds = clf.predict(grid_scaled).reshape(xx1.shape)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.contourf(xx1, xx2, preds, alpha=0.25, cmap="coolwarm", levels=[-0.5, 0.5, 1.5])
    sc = ax.scatter(X[:, 0], X[:, 1], c=labels, cmap="coolwarm", edgecolor="k", s=80)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_title(f"Function {func_id} -- SVM ({kernel}) decision boundary\n(blue=low output, red=high output)")
    plt.tight_layout()

    if save:
        os.makedirs("eda_plots", exist_ok=True)
        path = f"eda_plots/function_{func_id}_svm_boundary.png"
        plt.savefig(path, dpi=120)
        print(f"Saved: {path}")
    plt.show()


### Run the SVM analysis

In [ ]:
svm_kernel_comparison_all()

In [ ]:
svm_feature_importance_all()

### SVM plots

In [ ]:
plot_kernel_comparison()

In [ ]:
plot_feature_importance_all()

In [ ]:
plot_decision_boundary(1)
plot_decision_boundary(2)

## 9. A third model, to measure how complicated each function is

### The question this answers

Section 8 asked whether each function is simple or complicated using a classifier. This
section asks the same question a different way, and the two answers agreeing is more
convincing than either alone.

### The method

I fit two models to the same data and compare how well each one explains it. The first is
a straight line relationship: score equals some weighted sum of the inputs. The second is
a small neural network, which can represent curves and interactions that a straight line
cannot.

If the neural network fits dramatically better, the function has real structure that a
simple model cannot capture. If the two are similar, a simple description is good enough.
Three of the eight functions showed a large gap, three showed almost none.

### What the gradients add

The second command estimates, at each function's best known point, how sensitive the
score is to nudging each input slightly. A large value means that input matters a lot
right there. This is a local view rather than a global one, which is a useful corrective:
an input can matter enormously in one region and barely at all in another.

### An honest caveat about this section

With only a few dozen observations per function, a neural network is prone to memorising
noise rather than learning structure. I tested this directly: with the library's default
settings, one function's fit was actively worse than useless, and tightening a single
setting fixed it. The lesson is that the **comparison** between the two models is
meaningful, while the absolute quality of either is not. I would not trust this network
to predict anything.


In [ ]:
"""
Neural network surrogate analysis of the BBO capstone functions
========================================================================
Trains a small neural network (MLPRegressor) as a surrogate model for each
function's observed data, then:
- Estimates gradients via finite differences on the trained surrogate, to see
  which input dimensions the surrogate is most sensitive to (a numerical
  analogue of what backpropagation computes internally)
- Compares the neural network's fit quality (R^2) against a plain linear
  regression, as a way to quantify how much non-linearity the data seems to
  contain

This is a secondary, exploratory tool -- it does NOT replace the Gaussian
Process used for actual query selection. With only a handful of points per
function, a neural network is prone to overfitting, so results should be read
as rough signals, not precise ground truth.

Requirements: numpy, scikit-learn
Usage: place this file next to bbo_optimizer.py (it reuses its data-loading
logic) and import the functions in the notebook.
"""

import warnings
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

from bbo_optimizer import _load_history, FUNCTION_DIMS


def _prepare_regression_data(func_id):
    """Load all (initial + weekly) points for a function as (X, y) for regression."""
    X, y = _load_history(func_id)
    return np.array(X), np.array(y)


def train_surrogate(func_id, hidden_layer_sizes=(16, 8), max_iter=3000, random_state=42, alpha=1.0):
    """
    Train a small MLPRegressor as a surrogate model for a function, on
    standardised inputs and outputs. Returns the fitted model and the scalers
    used, so they can be reused for gradient estimation.

    alpha (L2 regularisation strength) defaults to 1.0 rather than sklearn's
    default 0.0001: a hyperparameter experiment on this project's own data
    (Functions 4 and 8) showed the default under-regularises with only 24-44
    points available, causing unstable/overfit fits (e.g. Function 4's CV R^2
    went from -0.100 at alpha=0.0001 to +0.145 at alpha=1.0). Pass a smaller
    alpha explicitly if you have enough data to prefer more flexibility.
    """
    X, y = _prepare_regression_data(func_id)
    n_points = len(y)
    dim = FUNCTION_DIMS[func_id]

    if n_points < 8:
        print(f"Function {func_id}: only {n_points} points, too few to train a reliable surrogate.")
        return None

    y_range = y.max() - y.min()
    if y_range < 0.01:
        print(f"Function {func_id}: output range is only {y_range:.2e} across all {n_points} observed "
              f"points (essentially flat/near-zero) -- no real signal yet to fit a surrogate to.")
        return None

    x_scaler = StandardScaler().fit(X)
    y_scaler = StandardScaler().fit(y.reshape(-1, 1))
    X_scaled = x_scaler.transform(X)
    y_scaled = y_scaler.transform(y.reshape(-1, 1)).ravel()

    model = MLPRegressor(
        hidden_layer_sizes=hidden_layer_sizes,
        activation="relu",
        alpha=alpha,
        max_iter=max_iter,
        random_state=random_state,
        early_stopping=(n_points >= 20),
        validation_fraction=0.2,
    )
    model.fit(X_scaled, y_scaled)
    return model, x_scaler, y_scaler


def surrogate_gradients(func_id, epsilon=1e-3):
    """
    Estimate the surrogate's gradient at the best observed point via finite
    differences: for each input dimension, perturb it slightly and see how
    much the surrogate's predicted output changes. Larger |gradient| suggests
    the surrogate has learned that dimension matters more, at least locally
    around the current best point -- a numerical proxy for what
    backpropagation computes internally during training.
    """
    result = train_surrogate(func_id)
    if result is None:
        return None
    model, x_scaler, y_scaler = result

    X, y = _prepare_regression_data(func_id)
    dim = FUNCTION_DIMS[func_id]
    best_idx = int(np.argmax(y))
    x0 = X[best_idx].copy()

    gradients = np.zeros(dim)
    for d in range(dim):
        x_plus, x_minus = x0.copy(), x0.copy()
        x_plus[d] = min(x_plus[d] + epsilon, 1.0)
        x_minus[d] = max(x_minus[d] - epsilon, 0.0)

        y_plus = y_scaler.inverse_transform(
            model.predict(x_scaler.transform(x_plus.reshape(1, -1))).reshape(-1, 1))[0, 0]
        y_minus = y_scaler.inverse_transform(
            model.predict(x_scaler.transform(x_minus.reshape(1, -1))).reshape(-1, 1))[0, 0]

        denom = x_plus[d] - x_minus[d]
        gradients[d] = (y_plus - y_minus) / denom if denom > 0 else 0.0

    order = np.argsort(np.abs(gradients))[::-1]
    print(f"\nFunction {func_id} ({dim}-D) -- estimated surrogate gradient at best point so far:")
    for rank, idx in enumerate(order, start=1):
        print(f"  #{rank}: x{idx + 1}  gradient={gradients[idx]:+.3f}")
    return gradients


def surrogate_gradients_all():
    """Run gradient estimation for all 8 functions."""
    for func_id in FUNCTION_DIMS:
        surrogate_gradients(func_id)


def compare_linear_vs_nn(func_id, cv_folds=5):
    """
    Compare a plain linear regression against the neural network surrogate
    using cross-validated R^2, to quantify how much non-linear structure the
    data seems to contain. A much higher R^2 for the neural network suggests
    real non-linearity the linear model can't capture; similar R^2 suggests
    the linear model is already a reasonable approximation (at least with the
    current amount of data).
    """
    X, y = _prepare_regression_data(func_id)
    n_points = len(y)
    dim = FUNCTION_DIMS[func_id]

    if n_points < 8:
        print(f"Function {func_id}: only {n_points} points, too few for a reliable comparison.")
        return None

    y_range = y.max() - y.min()
    if y_range < 0.01:
        print(f"\nFunction {func_id} ({dim}-D) -- output range is only {y_range:.2e} across all "
              f"{n_points} observed points (essentially flat/near-zero). R^2 is not meaningful here "
              f"and can blow up numerically on such small-scale data -- both models trivially predict "
              f"~0, since there is no real variation to explain yet.")
        return None

    x_scaler = StandardScaler().fit(X)
    y_scaler = StandardScaler().fit(y.reshape(-1, 1))
    X_scaled = x_scaler.transform(X)
    y_scaled = y_scaler.transform(y.reshape(-1, 1)).ravel()

    folds = max(2, min(cv_folds, n_points // 3))

    lin_scores = cross_val_score(LinearRegression(), X_scaled, y_scaled, cv=folds, scoring="r2")

    nn_model = MLPRegressor(hidden_layer_sizes=(16, 8), alpha=1.0, max_iter=3000, random_state=42,
                             early_stopping=(n_points >= 20), validation_fraction=0.2)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        nn_scores = cross_val_score(nn_model, X_scaled, y_scaled, cv=folds, scoring="r2")

    print(f"\nFunction {func_id} ({dim}-D) -- {n_points} points, {folds}-fold CV R^2:")
    print(f"  Linear regression -> R^2 = {lin_scores.mean():.3f} (+/- {lin_scores.std():.3f})")

    if np.isnan(nn_scores).any():
        print(f"  Neural network    -> could not be reliably evaluated with only {n_points} points "
              f"(too few for stable training within cross-validation folds)")
        return {"linear_r2": lin_scores.mean(), "nn_r2": None}

    print(f"  Neural network    -> R^2 = {nn_scores.mean():.3f} (+/- {nn_scores.std():.3f})")
    gap = nn_scores.mean() - lin_scores.mean()
    if gap > 0.15:
        print(f"  -> Large gap ({gap:+.3f}): strong evidence of non-linear structure")
    elif gap > 0.05:
        print(f"  -> Moderate gap ({gap:+.3f}): some non-linear structure")
    else:
        print(f"  -> Small gap ({gap:+.3f}): linear model already competitive")

    return {"linear_r2": lin_scores.mean(), "nn_r2": nn_scores.mean()}


def compare_linear_vs_nn_all():
    """Run the linear-vs-neural-network comparison for all 8 functions."""
    for func_id in FUNCTION_DIMS:
        compare_linear_vs_nn(func_id)


### Run the neural network surrogate analysis

In [ ]:
compare_linear_vs_nn_all()

In [ ]:
surrogate_gradients_all()

## 10. The most interesting thing I found

### What happened

One of the eight functions behaves in a way I did not expect, and working it out was
worth more than several weeks of ordinary queries.

I noticed that two completely different inputs had returned **exactly** the same score,
to the last decimal: 1616.6425. That is not a coincidence. It meant the function was
treating those two inputs as equivalent, even though they sit far apart in any normal
sense of distance.

### The pattern

The score does not depend on where the inputs sit. It depends on **how many** of the four
inputs are pushed all the way to their maximum value of 1. It does not matter which ones.

| Inputs at the maximum | Score |
|---|---|
| none | about 163 |
| one | about 191 |
| two | about 1617 |
| three | about 4441 |
| all four | 8662 |

I tested each of these cases deliberately rather than waiting for the optimiser to
stumble on them, overriding its suggestion for three rounds in a row to do so.

### Why this matters

Two things stand out. The jump between one input at the maximum and two is roughly
eightfold, so there is a threshold rather than a gradual climb. And a point sitting
exactly in the middle of the space scored 32, which is lower than the all zeros case, so
the function rewards extremes in either direction and penalises the middle ground.

More broadly, this is the clearest example in the project of a deliberately designed
question beating blind optimisation. The optimiser had already found the best corner and
kept proposing it. It would never have mapped the pattern, because mapping a pattern is
not what it is for.

### What the cell below draws

A chart with the number of inputs at the maximum along the bottom and the score up the
side. The starting examples appear in grey, my weekly queries in red with their values
labelled. The staircase shape is the finding, visible at a glance.


In [ ]:
"""
Function 5 symmetry analysis
========================================================================
Plots the relationship between the number of input coordinates set to 1.0
and the observed output y, for Function 5. Across several weekly queries,
the output appeared to depend only on the COUNT of coordinates at the
extreme value 1.0, not on which specific coordinates were set -- e.g.
x=[0,0,1,1] and x=[1,1,0,0] both returned y=1616.6425.

Requirements: numpy, matplotlib
Usage: place this file next to bbo_optimizer.py and import in the notebook.
"""

import numpy as np
import matplotlib.pyplot as plt
import os

from bbo_optimizer import _load_history


def plot_function5_symmetry(save=True, tolerance=0.001):
    """
    Scatter plot of "number of coordinates == 1.0" vs observed output y for
    Function 5, using all data (initial + weekly). Points with the same count
    are jittered slightly on the x-axis so repeated values are visible rather
    than overlapping exactly.
    """
    X, y = _load_history(5)
    X, y = np.array(X), np.array(y)

    counts = (X >= (1.0 - tolerance)).sum(axis=1)

    fig, ax = plt.subplots(figsize=(8, 5))

    rng = np.random.RandomState(0)
    jitter = rng.uniform(-0.08, 0.08, size=len(counts))

    colors = ["gray" if c == 0 else "crimson" for c in counts]
    is_weekly_like = counts > 0  # count>0 points happen to all be weekly queries here

    ax.scatter(counts[counts == 0] + jitter[counts == 0], y[counts == 0],
               alpha=0.5, color="gray", label="Initial data (0 coords at 1.0)", s=50)
    for c in sorted(set(counts[counts > 0])):
        mask = counts == c
        ax.scatter(counts[mask] + jitter[mask], y[mask],
                   color="crimson", s=120, edgecolor="k", zorder=3)

    # Annotate the distinct weekly points with their exact y value
    for c, yv in zip(counts[counts > 0], y[counts > 0]):
        ax.annotate(f"{yv:.1f}", (c, yv), textcoords="offset points",
                   xytext=(8, 4), fontsize=9, color="crimson")

    ax.scatter([], [], color="crimson", s=120, edgecolor="k", label="Weekly queries (coords at 1.0)")

    ax.set_xlabel("Number of input coordinates set to 1.0")
    ax.set_ylabel("Output y")
    ax.set_xticks([0, 1, 2, 3, 4])
    ax.set_title("Function 5 -- output depends on COUNT of coordinates at 1.0,\nnot which specific ones (symmetry pattern)")
    ax.legend()
    plt.tight_layout()

    if save:
        os.makedirs("eda_plots", exist_ok=True)
        path = "eda_plots/function_5_symmetry.png"
        plt.savefig(path, dpi=120)
        print(f"Saved: {path}")
    plt.show()

    print("\nSummary by count of coordinates at 1.0:")
    for c in sorted(set(counts)):
        vals = y[counts == c]
        print(f"  count={c}: n={len(vals)}, y range=[{vals.min():.2f}, {vals.max():.2f}], mean={vals.mean():.2f}")


In [ ]:
plot_function5_symmetry()

## 11. Final results

The project is now complete: thirteen weekly queries have been submitted and recorded
for each of the eight functions. This section prints one closing table summarising the
whole thing.

### How to read the table

| Column | Meaning |
|---|---|
| **Best initial** | The highest score already present in the starting data provided by the course. This is the bar each weekly query had to beat. |
| **Best reached** | The highest score across everything, starting data and weekly queries combined. |
| **Record week** | Which of the thirteen weekly rounds produced that record. If it says `initial`, no weekly query ever beat the starting data for that function. |
| **Improved** | Whether the weekly queries improved on the starting position at all. |

### Why this table is worth looking at

The headline number, how high a score got, is less informative than **when** the record
arrived. A record set in week 2 and never beaten afterwards suggests the search found a
good region early and then stalled. A record set in week 11 or 12 suggests the method
was still making progress when the budget ran out, which raises the question of what
more rounds would have found.

The `Best initial` column matters too. A large gap between the starting bar and the
final score means the optimisation did real work. A small gap means the starting data
was already close to as good as this search could get.


In [ ]:
final_summary()